# State of Data Brasil — Análise de Gênero (2021–2024)

## Seção 0 — Imports e Carregamento da base de dados

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr, norm
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (classification_report, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
import shap
from fairlearn.metrics import (MetricFrame, demographic_parity_difference,
                                equalized_odds_difference)

os.makedirs('figures', exist_ok=True)
os.makedirs('data', exist_ok=True)

import warnings
warnings.filterwarnings('ignore')

SEED  = 42
np.random.seed(SEED)
ALPHA = 0.05
print('OK')

: 

In [ ]:
dfs_raw = {
    '2021': pd.read_csv('data/state_of_data_2021.csv', low_memory=False),
    '2022': pd.read_csv('data/state_of_data_2022.csv', low_memory=False),
    '2023': pd.read_csv('data/state_of_data_2023.csv', low_memory=False),
    '2024': pd.read_csv('data/state_of_data_2024.csv', low_memory=False),
}
for ano, df_raw in dfs_raw.items():
    print(f'{ano}: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas')

### 0.1 Mapeamento de colunas por ano

In [ ]:
COLUMN_MAPPING = {
    '2021': {
        'idade':            "('P1_a ', 'Idade')",
        'faixa_idade':      "('P1_a_a ', 'Faixa idade')",
        'genero':           "('P1_b ', 'Genero')",
        'estado':           "('P1_e ', 'Estado onde mora')",
        'uf':               "('P1_e_a ', 'uf onde mora')",
        'regiao':           "('P1_e_b ', 'Regiao onde mora')",
        'regiao_origem':    "('P1_g_b ', 'Regiao de origem')",
        'nivel_ensino':     "('P1_h ', 'Nivel de Ensino')",
        'area_formacao':    "('P1_i ', 'Área de Formação')",
        'situacao':         "('P2_a ', 'Qual sua situação atual de trabalho?')",
        'setor':            "('P2_b ', 'Setor')",
        'cargo':            "('P2_f ', 'Cargo Atual')",
        'gestor':           "('P2_d ', 'Gestor?')",
        'nivel':            "('P2_g ', 'Nivel')",
        'faixa_salarial':   "('P2_h ', 'Faixa salarial')",
        'tempo_area_dados': "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
        'tempo_area_ti':    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')",
        'modalidade':       "('P2_q ', 'Atualmente qual a sua forma de trabalho?')",
    },
    '2022': {
        'idade':            "('P1_a ', 'Idade')",
        'faixa_idade':      "('P1_a_1 ', 'Faixa idade')",
        'genero':           "('P1_b ', 'Genero')",
        'estado':           "('P1_i ', 'Estado onde mora')",
        'uf':               "('P1_i_1 ', 'uf onde mora')",
        'regiao':           "('P1_i_2 ', 'Regiao onde mora')",
        'regiao_origem':    "('P1_k ', 'Regiao de origem')",
        'nivel_ensino':     "('P1_l ', 'Nivel de Ensino')",
        'area_formacao':    "('P1_m ', 'Área de Formação')",
        'situacao':         "('P2_a ', 'Qual sua situação atual de trabalho?')",
        'setor':            "('P2_b ', 'Setor')",
        'cargo':            "('P2_f ', 'Cargo Atual')",
        'gestor':           "('P2_d ', 'Gestor?')",
        'nivel':            "('P2_g ', 'Nivel')",
        'faixa_salarial':   "('P2_h ', 'Faixa salarial')",
        'tempo_area_dados': "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
        'tempo_area_ti':    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')",
        'modalidade':       "('P2_p ', 'Atualmente qual a sua forma de trabalho?')",
    },
    '2023': {
        'idade':            "('P1_a ', 'Idade')",
        'faixa_idade':      "('P1_a_1 ', 'Faixa idade')",
        'genero':           "('P1_b ', 'Genero')",
        'estado':           "('P1_i ', 'Estado onde mora')",
        'uf':               "('P1_i_1 ', 'uf onde mora')",
        'regiao':           "('P1_i_2 ', 'Regiao onde mora')",
        'regiao_origem':    "('P1_k ', 'Regiao de origem')",
        'nivel_ensino':     "('P1_l ', 'Nivel de Ensino')",
        'area_formacao':    "('P1_m ', 'Área de Formação')",
        'situacao':         "('P2_a ', 'Qual sua situação atual de trabalho?')",
        'setor':            "('P2_b ', 'Setor')",
        'cargo':            "('P2_f ', 'Cargo Atual')",
        'gestor':           "('P2_d ', 'Gestor?')",
        'nivel':            "('P2_g ', 'Nivel')",
        'faixa_salarial':   "('P2_h ', 'Faixa salarial')",
        'tempo_area_dados': "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
        'tempo_area_ti':    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')",
        'modalidade':       "('P2_r ', 'Atualmente qual a sua forma de trabalho?')",
    },
    '2024': {
        'idade':            '1.a_idade',
        'faixa_idade':      '1.a.1_faixa_idade',
        'genero':           '1.b_genero',
        'estado':           '1.i_estado_onde_mora',
        'uf':               '1.i.1_uf_onde_mora',
        'regiao':           '1.i.2_regiao_onde_mora',
        'regiao_origem':    '1.k.2_regiao_de_origem',
        'nivel_ensino':     '1.l_nivel_de_ensino',
        'area_formacao':    '1.m_área_de_formação',
        'situacao':         '2.a_situação_de_trabalho',
        'setor':            '2.b_setor',
        'cargo':            '2.f_cargo_atual',
        'gestor':           '2.d_atua_como_gestor',
        'nivel':            '2.g_nivel',
        'faixa_salarial':   '2.h_faixa_salarial',
        'tempo_area_dados': '2.i_tempo_de_experiencia_em_dados',
        'tempo_area_ti':    '2.j_tempo_de_experiencia_em_ti',
        'modalidade':       '2.r_modelo_de_trabalho_atual',
    }
}

IA_COLS_RAW = {
    '2023': {
        'ia_prioridade_empresa':      "('P3_e ', 'AI Generativa é uma prioridade em sua empresa?')",
        'ia_tipos_uso_gestao':        "('P3_f ', 'Tipos de uso de AI Generativa e LLMs na empresa')",
        'ia_motivos_nao_usar':        "('P3_g ', 'Motivos que levam a empresa a não usar AI Genrativa e LLMs')",
        'ia_tipo_uso_pessoal':        "('P4_l ', 'Qual o tipo de uso de AI Generativa e LLMs na empresa')",
        'ia_usa_chatgpt_no_trabalho': "('P4_m ', 'Utiliza ChatGPT ou LLMs no trabalho?')",
    },
    '2024': {
        'ia_prioridade_empresa':      '3.e_ai_generativa_e_llm_é_uma_prioridade?',
        'ia_tipos_uso_gestao':        '3.f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa',
        'ia_motivos_nao_usar':        '3.g_motivos_para_não_usar_ai_generativa_e_llm',
        'ia_tipo_uso_pessoal':        '4.l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa',
        'ia_usa_chatgpt_no_trabalho': '4.m_usa_chatgpt_ou_copilot_no_trabalho?',
    },
}


### 0.2 Mapeamentos ordinais e de categorias

In [ ]:
FAIXA_SALARIAL_MAP = {
    'de R$ 1.001/mês a R$ 2.000/mês':   'R$1k-2k',
    'de R$ 2.001/mês a R$ 3.000/mês':   'R$2k-3k',
    'de R$ 2.001/mês a R$ 3000/mês':    'R$2k-3k',
    'de R$ 3.001/mês a R$ 4.000/mês':   'R$3k-4k',
    'de R$ 4.001/mês a R$ 6.000/mês':   'R$4k-6k',
    'de R$ 6.001/mês a R$ 8.000/mês':   'R$6k-8k',
    'de R$ 8.001/mês a R$ 12.000/mês':  'R$8k-12k',
    'de R$ 12.001/mês a R$ 16.000/mês': 'R$12k-16k',
    'de R$ 16.001/mês a R$ 20.000/mês': 'R$16k-20k',
    'de R$ 20.001/mês a R$ 25.000/mês': 'R$20k-25k',
    'de R$ 25.001/mês a R$ 30.000/mês': 'R$25k-30k',
    'de R$ 30.001/mês a R$ 40.000/mês': 'R$30k-40k',
    'Acima de R$ 40.001/mês':           'R$40k+',
}
FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k', 'R$2k-3k', 'R$3k-4k', 'R$4k-6k', 'R$6k-8k',
    'R$8k-12k', 'R$12k-16k', 'R$16k-20k', 'R$20k-25k',
    'R$25k-30k', 'R$30k-40k', 'R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal', 'Estudante de Graduação',
    'Graduação/Bacharelado', 'Especialização Lato Sensu',
    'Mestrado', 'Doutorado ou Phd',
]
#NIVEL_SENIORIDADE_ORDEM = ['Júnior', 'Pleno', 'Sênior', 'Gestor']
NIVEL_SENIORIDADE_ORDEM = ['Júnior', 'Pleno', 'Sênior']  # leakage
NIVEL_SENIORIDADE_EDA    = ['Júnior', 'Pleno', 'Sênior', 'Gestor']  # EDA

CARGO_MAP = {
    'Analista de Dados/Data Analyst':                    'Analista de Dados',
    'Analista de BI/BI Analyst':                         'Analista de Dados',
    'Analista de BI/BI Analyst/Analytics Engineer':      'Analista de Dados',
    'Analista de Negócios/Business Analyst':             'Analista de Dados',
    'Analista de Inteligência de Mercado/Market Intelligence': 'Analista de Dados',
    'Analista de Marketing':                             'Analista de Dados',
    'Analista Administrativo':                           'Analista de Dados',
    'Estatístico':                                       'Analista de Dados',
    'Economista':                                        'Analista de Dados',
    'Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect': 'Engenheiro de Dados',
    'Engenheiro de Dados/Data Engineer':                 'Engenheiro de Dados',
    'Analytics Engineer':                                'Engenheiro de Dados',
    'Arquiteto de Dados':                                'Engenheiro de Dados',
    'Arquiteto de dados':                                'Engenheiro de Dados',
    'DBA/Administrador de Banco de Dados':               'Engenheiro de Dados',
    'Engenheiro de Dados/Data Engineer/Data Architect':  'Engenheiro de Dados',
    'Arquiteto de Dados/Data Architect':                 'Engenheiro de Dados',
    'Cientista de Dados/Data Scientist':                 'Cientista de Dados',
    'Engenheiro de Machine Learning/ML Engineer':        'Cientista de Dados',
    'Engenheiro de Machine Learning/ML Engineer/AI Engineer': 'Cientista de Dados',
    'Professor':                                         'Professor/Pesquisador',
    'Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO)': 'Product Manager',
    'Product Manager/ Product Owner (PM/APM/DPM/GPM/PO)': 'Product Manager',
    'Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas': 'Desenvolvedor',
    'Desenvolvedor ou Engenheiro de Software':           'Desenvolvedor',
    'Analista de Sistemas/Analista de TI':               'Desenvolvedor',
    'Analista de Suporte/Analista Técnico':              'Desenvolvedor',
    'Suporte Técnico':                                   'Desenvolvedor',
    'Técnico':                                           'Desenvolvedor',
    'Outra Opção':                                       'Outro',
    'Outras Engenharias (não inclui dev)':               'Outro',
}
AREA_FORMACAO_MAP = {
    'Computação / Engenharia de Software / Sistemas de Informação/ TI': 'Computação / TI',
    'Outras Engenharias': 'Engenharia (outras)',
    'Outras Engenharias (não incluir engenharia de software ou TI)': 'Engenharia (outras)',
    'Economia/ Administração / Contabilidade / Finanças/ Negócios': 'Economia / Adm / Finanças',
    'Economia/ Administração / Contabilidade / Finanças': 'Economia / Adm / Finanças',
    'Estatística/ Matemática / Matemática Computacional/ Ciências Atuariais': 'Estatística / Matemática',
    'Estatística/ Matemática / Matemática Computacional': 'Estatística / Matemática',
    'Ciências Biológicas/ Farmácia/ Medicina/ Área da Saúde': 'Ciências da Saúde',
    'Ciências Biológicas/Farmácia/Medicina/Área da Saúde': 'Ciências da Saúde',
    'Marketing / Publicidade / Comunicação / Jornalismo': 'Marketing / Comunicação',
    'Marketing / Publicidade / Comunicação / Jornalismo / Ciências Sociais': 'Marketing / Comunicação',
    'Outra opção': 'Outras',
}
SITUACAO_MAP = {
    'Vivo no Brasil e trabalho remoto para empresa de fora do Brasil':      'Remoto p/ exterior',
    'Vivo no Brasil e trabalho remoto para empresa de fora do Brasil (PJ)': 'Remoto p/ exterior',
    'Desempregado, buscando recolocação':                                   'Desempregado',
    'Desempregado e não estou buscando recolocação':                        'Desempregado',
    'Somente Estudante (graduação)':                                        'Somente Estudante',
    'Somente Estudante (pós-graduação)':                                    'Somente Estudante',
    'Prefiro não informar':                                                  None,
}

REGIAO_MAP = {
    'Centro_Oeste': 'Centro-Oeste', 'Centro Oeste': 'Centro-Oeste',
    'centro-oeste': 'Centro-Oeste', 'Centro-oeste': 'Centro-Oeste',
    'nordeste': 'Nordeste', 'norte': 'Norte',
    'sudeste': 'Sudeste',  'sul':    'Sul',
    'Exterior': None, 'Prefiro não informar': None,
}

MODALIDADE_MAP = {
    'Modelo 100% presencial': 'Presencial',
    'Modelo 100% remoto': 'Remoto',
    'Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)': 'Híbrido flexível',
    'Modelo híbrido com dias fixos de trabalho presencial': 'Híbrido fixo',
}

ANOS     = ['2021', '2022', '2023', '2024']
REGIOES = ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte']

CORES_GENERO = {'Masculino': '#517493', 'Feminino': '#64313E'}
CORES_REGIAO = {
    'Sudeste':      '#2563EB',
    'Sul':          '#16A34A',
    'Nordeste':     '#EA580C',
    'Centro-Oeste': '#9333EA',
    'Norte':        '#CA8A04',
}
TEMPLATE = 'plotly_white'

In [ ]:
# ia_usa_chatgpt_no_trabalho
_NAO_USA       = 'Não utilizo nenhum tipo de solução de IA Generativa'
_USA_GRATUITA  = 'Utilizo apenas soluções gratuitas'
_USA_PAGA_PROP = 'pago do meu próprio bolso'
_USA_PAGA_EMP  = 'a empresa em que trabalho paga'
_USA_COPILOT   = 'Copilot'

# ia_prioridade_empresa
_PRIORIDADE_ALTA = [
    'Sim, é nossa principal prioridade',
    'Sim, está entre nossas principais prioridades',
]

# ia_tipo_uso_pessoal / ia_tipos_uso_gestao
_ORG_COLABORADORES = 'Colaboradores utilizando soluções baseadas em AI Generativa'


def engenharia_ia(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria features binárias relacionadas ao uso individual e organizacional
    de IA generativa, preservando a distinção entre ausência de resposta
    e pergunta inexistente no ano da pesquisa.
    """
    df = df.copy()

    # ------------------------------------------------------------------
    # 1. Uso individual de IA
    # ------------------------------------------------------------------

    col = 'ia_usa_chatgpt_no_trabalho'

    flags_uso = {
        'ia_nao_usa':       _NAO_USA,
        'ia_usa_gratuita':  _USA_GRATUITA,
        'ia_usa_paga_prop': _USA_PAGA_PROP,
        'ia_usa_paga_emp':  _USA_PAGA_EMP,
        'ia_usa_copilot':   _USA_COPILOT,
    }

    s = df[col].fillna('')
    sem_dados = df[col].isna()

    for nova_col, padrao in flags_uso.items():
        df[nova_col] = s.str.contains(padrao, na=False).astype('Int8')
        df.loc[sem_dados, nova_col] = pd.NA

    # ------------------------------------------------------------------
    # 2. IA como prioridade estratégica da empresa
    # ------------------------------------------------------------------

    col_pri = 'ia_prioridade_empresa'

    if col_pri in df.columns:
        s_pri = df[col_pri].fillna('')

        df['ia_empresa_prioridade_alta'] = (
            s_pri.apply(
                lambda x: (
                    1 if any(p in x for p in _PRIORIDADE_ALTA)
                    else pd.NA if x == ''
                    else 0
                )
            )
            .astype('Int8')
        )
    else:
        df['ia_empresa_prioridade_alta'] = pd.Series(pd.NA, index=df.index, dtype='Int8')

    # ------------------------------------------------------------------
    # 3. Uso difuso na organização
    # ------------------------------------------------------------------

    col_tp = 'ia_tipo_uso_pessoal'
    col_tg = 'ia_tipos_uso_gestao'

    s_org = pd.Series('', index=df.index)

    if col_tp in df.columns:
        s_org += df[col_tp].fillna('')

    if col_tg in df.columns:
        s_org += df[col_tg].fillna('')

    sem_org = (
        df.get(col_tp, pd.Series(pd.NA, index=df.index)).isna()
        &
        df.get(col_tg, pd.Series(pd.NA, index=df.index)).isna()
    )

    df['ia_org_uso_difuso'] = (
        s_org.str.contains(_ORG_COLABORADORES, na=False)
        .astype('Int8')
    )

    df.loc[sem_org, 'ia_org_uso_difuso'] = pd.NA

    return df

In [ ]:
IA_FEATURES_MODELO_B = [
    'ia_nao_usa',
    'ia_usa_gratuita',
    'ia_usa_paga_prop',
    'ia_usa_paga_emp',
    'ia_usa_copilot',
    'ia_empresa_prioridade_alta',
    'ia_org_uso_difuso',
]

In [ ]:
# Labels legíveis para SHAP / gráficos
IA_FEATURE_LABELS = {
    'ia_nao_usa':                  'Não usa IA',
    'ia_usa_gratuita':             'Usa IA gratuita',
    'ia_usa_paga_prop':            'Usa IA paga (bolso próprio)',
    'ia_usa_paga_emp':             'Usa IA paga (empresa paga)',
    'ia_usa_copilot':              'Usa Copilot',
    'ia_empresa_prioridade_alta':  'IA é prioridade alta na empresa',
    'ia_org_uso_difuso':           'Uso difuso na organização',
}

---
## Seção 1 — Pipeline de pré-processamento

In [ ]:
def build_year_dataframe(df_raw: pd.DataFrame, year: str) -> pd.DataFrame:
    base_map = COLUMN_MAPPING[year]
    cols_base = {old: new for new, old in base_map.items() if old in df_raw.columns}
    df = df_raw[list(cols_base.keys())].rename(columns=cols_base).copy()

    if year in IA_COLS_RAW:
        ia_map = IA_COLS_RAW[year]
        cols_ia = {old: new for new, old in ia_map.items() if old in df_raw.columns}
        df = pd.concat([df, df_raw[list(cols_ia.keys())].rename(columns=cols_ia)], axis=1)

    df.insert(0, 'ano_pesquisa', year)
    return df


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df['genero'].isin(['Masculino', 'Feminino'])].copy()

    df['nivel_ensino'] = df['nivel_ensino'].replace({'Pós-graduação': 'Especialização Lato Sensu', 'Prefiro não informar': np.nan})
    df['nivel_ensino'] = pd.Categorical(df['nivel_ensino'], categories=NIVEL_ENSINO_ORDEM, ordered=True)

    df['faixa_salarial'] = df['faixa_salarial'].str.replace(r'\s+', ' ', regex=True).str.strip().replace(FAIXA_SALARIAL_MAP)
    df['faixa_salarial'] = pd.Categorical(df['faixa_salarial'], categories=FAIXA_SALARIAL_ORDEM, ordered=True)

    df['cargo']        = df['cargo'].replace(CARGO_MAP)
    df['area_formacao']= df['area_formacao'].replace(AREA_FORMACAO_MAP)
    df['regiao']       = df['regiao'].replace(REGIAO_MAP)
    df['modalidade']   = df['modalidade'].map(MODALIDADE_MAP).fillna(df['modalidade'])

    df['nivel'] = pd.Categorical(
        df['nivel'].where(df['nivel'].isin(NIVEL_SENIORIDADE_ORDEM)),
        categories=NIVEL_SENIORIDADE_ORDEM, ordered=True
    )

    # NaN mantido: distinto de "não é gestor"
    df['gestor'] = df['gestor'].replace({'Sim': 1, 'Não': 0, 'sim': 1, 'não': 0}).pipe(pd.to_numeric, errors='coerce')

    df['situacao'] = df['situacao'].replace(SITUACAO_MAP)
    df = df[df['situacao'].notna()]

    return df.reset_index(drop=True)

def criar_targets_salariais(df: pd.DataFrame) -> pd.DataFrame:
    """Cria versões agrupadas da faixa salarial para experimento de sensibilidade."""
    df = df.copy()

    ALTA   = ['R$12k-16k', 'R$16k-20k', 'R$20k-25k', 'R$25k-30k', 'R$30k-40k', 'R$40k+']
    MEDIA  = ['R$6k-8k', 'R$8k-12k']
    # Baixa = demais

    def agrupa_3(fx):
        if pd.isna(fx):   return np.nan
        if fx in ALTA:    return 'Alta'
        if fx in MEDIA:   return 'Média'
        return 'Baixa'

    df['faixa_salarial_3']   = df['faixa_salarial'].apply(agrupa_3)
    df['faixa_salarial_bin'] = (df['faixa_salarial'].isin(ALTA)).astype('Int8')
    df.loc[df['faixa_salarial'].isna(), 'faixa_salarial_bin'] = pd.NA

    return df


def run_pipeline(dfs_raw: dict) -> pd.DataFrame:
    frames = [build_year_dataframe(dfs_raw[year], year) for year in ANOS]
    df = pd.concat(frames, ignore_index=True)
    df = preprocess(df)
    df = engenharia_ia(df)
    return df


df = run_pipeline(dfs_raw)
print(f'Dataset completo: {df.shape}')

In [ ]:
# Split temporal: treino (2021–2023) e teste (2024)
df_train = df[df['ano_pesquisa'] != '2024'].copy()
df_test  = df[df['ano_pesquisa'] == '2024'].copy()

# Persistência dos datasets
df.to_parquet('data/df_full.parquet', index=False)
df_train.to_parquet('data/df_train.parquet', index=False)
df_test.to_parquet('data/df_test.parquet', index=False)


print(
    f'Full: {df.shape} | '
    f'Treino: {df_train.shape} | '
    f'Teste: {df_test.shape}'
)

print(
    f'Cobertura IA — '
    f'Treino: {df_train[IA_FEATURES_MODELO_B].notna().any(axis=1).mean()*100:.1f}% | '
    f'Teste: {df_test[IA_FEATURES_MODELO_B].notna().any(axis=1).mean()*100:.1f}%'
)

---
## Seção 2 — Análise Exploratória de Dados EDA

Cada visualização responde uma pergunta diferente, todas longitudinais por ano sem agregação e sem redundância. 

- 2.1 Representatividade feminina ao longo dos anos
- 2.2 Senioridade por gênero
- 2.3 Distribuição de cargos por gênero
- 2.4 Diferença salarial entre gêneros
- 2.5 Escolaridade por gênero
- 2.6 Área de formação por gênero
- 2.7 Distribuição de Experiência na Área de Dados por Gênero
- 2.8 Modalidade de trabalho por gênero
- 2.9 Adoção de IA por gênero
- 2.10 Análise regional da participação feminina


### 2.0 Config

In [ ]:
def fmt(fig, title, height=450, **kw):
    fig.update_layout(
        title=title,
        template=TEMPLATE,
        height=height,
        legend=dict(orientation='h', y=1.02, x=1, xanchor='right'),
        **kw,
    )
    return fig
 
 
def dedup_legend(fig):
    seen = set()
    for t in fig.data:
        t.showlegend = t.name not in seen
        seen.add(t.name)
    return fig
 
 
def subplots_pct_fem(df, col, order, title, height=420):
    """
    Subplots 1×4 (um por ano): % feminino por categoria.
    - % do gênero feminino dentro de cada categoria sobreposto na barra (cor branca, bold)
    - n total da categoria exibido fora da barra
    - range do eixo X calculado dinamicamente (máx + margem de 15 pp)
    """
    # calcula range dinâmico
    tab_all = pd.crosstab(df[col], df['genero']).reindex(order).fillna(0)
    n_all   = tab_all.sum(axis=1)
    pct_max = (tab_all.get('Feminino', 0) / n_all.replace(0, np.nan) * 100).max()
    x_max   = min(100, np.ceil((pct_max + 15) / 10) * 10)
 
    fig = make_subplots(1, 4, subplot_titles=ANOS, shared_yaxes=True)
 
    for i, ano in enumerate(ANOS, 1):
        sub = df[df['ano_pesquisa'] == ano]
        tab = pd.crosstab(sub[col], sub['genero']).reindex(order).fillna(0)
        n   = tab.sum(axis=1)
        pct = tab.get('Feminino', pd.Series(0, index=tab.index)) / n.replace(0, np.nan) * 100
 
        # texto dentro da barra: % (branco, negrito visual via tamanho)
        text_inside = [
            f'<b>{v:.0f}%</b>' if not np.isnan(v) else ''
            for v in pct
        ]
        # texto fora da barra: n total
        text_outside = [f'n={int(ni)}' for ni in n]
 
        # barra principal com % dentro
        fig.add_trace(go.Bar(
            y=pct.index,
            x=pct.values,
            orientation='h',
            marker_color=CORES_GENERO['Feminino'],
            text=text_inside,
            textposition='inside',
            textfont=dict(color='white', size=11),
            showlegend=False,
            customdata=n.values,
            hovertemplate='%{y}: %{x:.1f}% feminino (n=%{customdata})<extra></extra>',
        ), row=1, col=i)
 
        # marcadores invisíveis para exibir n fora da barra
        fig.add_trace(go.Scatter(
            y=pct.index,
            x=pct.values,
            mode='text',
            text=text_outside,
            textposition='middle right',
            textfont=dict(size=9, color='gray'),
            showlegend=False,
        ), row=1, col=i)
 
        fig.update_xaxes(range=[0, x_max], ticksuffix='%', row=1, col=i)
 
    return fmt(fig, title, height=height)
 
 
def subplots_piramide(df, col, order, title, height=420):
    """
    Subplots 1×4: pirâmide bilateral — % dentro de cada gênero por categoria.
    Feminino à esquerda (negativo), Masculino à direita.
    Ordem da legenda: Feminino primeiro.
    """
    fig = make_subplots(1, 4, subplot_titles=ANOS, shared_yaxes=True)
 
    # Feminino sempre primeiro para fixar ordem da legenda
    for genero in ['Feminino', 'Masculino']:
        for i, ano in enumerate(ANOS, 1):
            sub = df[df['ano_pesquisa'] == ano]
            tab = pd.crosstab(sub[col], sub['genero']).reindex(order).fillna(0)
            pct = tab.div(tab.sum(), axis=1) * 100

            vals = pct.get(genero, pd.Series(0, index=pct.index))
            x    = -vals if genero == 'Feminino' else vals

            fig.add_trace(go.Bar(
                y=pct.index,
                x=x,
                orientation='h',
                name=genero,
                legendgroup=genero,
                marker_color=CORES_GENERO[genero],
                text=[f'{v:.0f}%' for v in vals],
                textposition='auto',
                textfont=dict(color='white', size=10),
            ), row=1, col=i)

    # Linha central e eixos aplicados uma vez por subplot, fora do loop de gênero
    for i in range(1, len(ANOS) + 1):
        fig.add_shape(
            type='line', x0=0, x1=0,
            y0=-0.5, y1=len(order) - 0.5,
            line=dict(color='black', dash='dot', width=1),
            row=1, col=i,
        )
        fig.update_xaxes(
            range=[-60, 60],
            tickvals=[-40, -20, 0, 20, 40],
            ticktext=['40%', '20%', '0', '20%', '40%'],
            row=1, col=i,
        )

    dedup_legend(fig)
    return fmt(fig, title, height=height, barmode='relative')

### 2.1 — Representatividade feminina ao longo dos anos



In [ ]:
tab = pd.crosstab(df['ano_pesquisa'], df['genero']).reindex(ANOS)
pct_fem = tab['Feminino'].div(tab.sum(axis=1)).mul(100)
 
fig = go.Figure(go.Scatter(
    x=ANOS,
    y=pct_fem,
    mode='lines+markers+text',
    text=[
        f'<b>{v:.1f}%</b><br><sup>n={int(n)}</sup>'
        for v, n in zip(pct_fem, tab.sum(axis=1))
    ],
    textposition='top center',
    line=dict(color=CORES_GENERO['Feminino'], width=3),
    marker=dict(size=10),
))
 
fig = fmt(fig, 'Representatividade Feminina por Ano (2021–2024)', height=360)
fig.update_yaxes(title='% Feminino', ticksuffix='%', range=[0, 40])
fig.show()


### 2.2 — Senioridade por gênero



In [ ]:
fig = subplots_piramide(
    df, 'nivel', NIVEL_SENIORIDADE_EDA,
    'Senioridade por Gênero (2021–2024)',
)
fig.show()


### 2.3 — Cargo por gênero

In [ ]:
CARGO_ORDEM = [
    'Analista de Dados',
    'Cientista de Dados',
    'Engenheiro de Dados',
    'Desenvolvedor',
    'Product Manager',
    'Professor/Pesquisador',
    'Outro',
]
 
fig = subplots_pct_fem(
    df, 'cargo', CARGO_ORDEM,
    '% Feminino por Cargo (2021–2024)',
    height=440,
)
fig.show()



### 2.4 — Distribuição salarial por gênero

In [ ]:
fig = subplots_piramide(
    df, 'faixa_salarial', FAIXA_SALARIAL_ORDEM,
    'Distribuição Salarial por Gênero — % dentro de cada gênero (2021–2024)',
    height=560,
)
fig.show()


### 2.5 — Escolaridade por gênero

In [ ]:
fig = subplots_pct_fem(
    df, 'nivel_ensino', NIVEL_ENSINO_ORDEM,
    '% Feminino por Nível de Ensino (2021–2024)',
    height=440,
)
fig.show()



In [ ]:
# Distribuição de escolaridade dentro de cada gênero (% por nível)
tab_ens = pd.crosstab(df['nivel_ensino'], df['genero'])
tab_ens_pct = tab_ens.div(tab_ens.sum(axis=0), axis=1) * 100
tab_ens_pct = tab_ens_pct.reindex(NIVEL_ENSINO_ORDEM)

fig = go.Figure()
for genero in ['Feminino', 'Masculino']:
    fig.add_trace(go.Bar(
        name=genero,
        x=tab_ens_pct.index,
        y=tab_ens_pct[genero],
        marker_color=CORES_GENERO[genero],
        text=[f"{v:.1f}%" for v in tab_ens_pct[genero]],
        textposition='outside',
    ))
fig = fmt(
    fig,
    'Distribuição de Escolaridade por Gênero (% dentro de cada gênero)',
    height=420,
    barmode='group',
)
fig.update_yaxes(title='%', ticksuffix='%')
fig.show()


### 2.6 — Área de formação por gênero

In [ ]:
AREA_ORDEM = [
    'Computação / TI',
    'Engenharia (outras)',
    'Estatística / Matemática',
    'Economia / Adm / Finanças',
    'Ciências da Saúde',
    'Marketing / Comunicação',
    'Outras',
]
 
fig = subplots_pct_fem(
    df, 'area_formacao', AREA_ORDEM,
    '% Feminino por Área de Formação (2021–2024)',
    height=440,
)
fig.show()


### 2.7 — Experiência na área de dados por gênero

In [ ]:
TEMPO_ORDEM = [
    'Menos de 1 ano',
    'de 1 a 2 anos',
    'de 2 a 3 anos',
    'de 3 a 4 anos',
    'de 4 a 5 anos',
    'de 5 a 6 anos',
    'de 6 a 7 anos',
    'de 7 a 10 anos',
    'Mais de 10 anos',
]
 
tempo_valido = [t for t in TEMPO_ORDEM if t in df['tempo_area_dados'].dropna().unique()]
 
fig = subplots_pct_fem(
    df[df['tempo_area_dados'].isin(tempo_valido)],
    'tempo_area_dados',
    tempo_valido,
    '% Feminino por Tempo de Experiência em Dados (2021–2024)',
    height=440,
)
fig.show()


### 2.8 — Modalidade de trabalho por gênero

In [ ]:
MODALIDADE_ORDEM = ['Remoto', 'Híbrido flexível', 'Híbrido fixo', 'Presencial']

fig = subplots_piramide(
    df[df['modalidade'].notna()], 'modalidade', MODALIDADE_ORDEM,
    '% Feminino por Modalidade de Trabalho (2021–2024)',
    height=400,
)
fig.show()

### 2.9 — Adoção de IA generativa por gênero 

In [ ]:
ANOS_IA = ['2023', '2024']
df_ia = df[df['ano_pesquisa'].isin(ANOS_IA)].copy()

df_ia['usa_ia_individual'] = df_ia['ia_nao_usa'].map(
    {1: 0, 0: 1, pd.NA: pd.NA}, na_action='ignore'
).astype('Int8')

df_ia['usa_ia_paga'] = (
    (df_ia['ia_usa_paga_prop'].fillna(0) | df_ia['ia_usa_paga_emp'].fillna(0))
    .where(df_ia['ia_usa_paga_prop'].notna() | df_ia['ia_usa_paga_emp'].notna())
    .astype('Int8')
)

n_tab = df_ia.groupby(['ano_pesquisa', 'genero']).size().reset_index(name='n')

def grafico_ia(col, titulo, ylabel):
    sub_valid = df_ia.dropna(subset=[col])
    tab = (
        sub_valid.groupby(['ano_pesquisa', 'genero'])[col]
        .mean().mul(100).reset_index()
        .merge(n_tab, on=['ano_pesquisa', 'genero'])
    )
    fig = go.Figure()
    for genero in ['Feminino', 'Masculino']:
        sub = tab[tab['genero'] == genero]
        fig.add_trace(go.Bar(
            x=sub['ano_pesquisa'],
            y=sub[col],
            name=genero,
            marker_color=CORES_GENERO[genero],
            text=[f'<b>{v:.1f}%</b><br><sup>n={n}</sup>'
                  for v, n in zip(sub[col], sub['n'])],
            textposition='outside',
        ))
    fig = fmt(fig, titulo, height=420, barmode='group')
    fig.update_yaxes(title=ylabel, ticksuffix='%', range=[0, 105])
    return fig

grafico_ia('ia_empresa_prioridade_alta',
           'Empresa Prioriza IA Generativa por Gênero (2023–2024)',
           '% que relata priorização').show()

grafico_ia('usa_ia_individual',
           'Uso Individual de IA Generativa por Gênero (2023–2024)',
           '% que usa IA individualmente').show()

grafico_ia('usa_ia_paga',
           'Uso de IA Paga por Gênero (2023–2024)',
           '% que usa IA paga').show()

### 2.10 — Representatividade feminina por região

In [ ]:
df_reg = df[df['regiao'].isin(REGIOES)].copy()

fig = go.Figure()
for regiao in REGIOES:
    pcts, textos = [], []
    for ano in ANOS:
        sub = df_reg[(df_reg['regiao'] == regiao) & (df_reg['ano_pesquisa'] == ano)]
        ct  = sub['genero'].value_counts()
        n   = ct.sum()
        fem = ct.get('Feminino', 0)
        v   = round(fem / n * 100, 1) if n >= 10 else None
        pcts.append(v)
        textos.append(f'{v}%' if v else 'n<10')
    fig.add_trace(go.Scatter(
        x=ANOS, y=pcts, mode='lines+markers+text', name=regiao,
        line=dict(color=CORES_REGIAO[regiao], width=2.5), marker=dict(size=9),
        text=textos, textposition='top center',
    ))
fmt(fig, '% Feminino por Região (2021–2024)', height=420)
fig.update_yaxes(ticksuffix='%', range=[0, 45])
fig.show()

# CRUZAMENTO AREA DE ORIGEM COM REGIAO - MIGROU?

---
## Seção 3 — Análise Estatística 

### 3.0 Funções auxiliares

In [ ]:
def cramers_v(ct: pd.DataFrame) -> float:

    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.to_numpy().sum()
    r, c = ct.shape
    phi2      = chi2 / n
    phi2_corr = max(0, phi2 - (r - 1) * (c - 1) / (n - 1))
    r_corr    = r - (r - 1) ** 2 / (n - 1)
    c_corr    = c - (c - 1) ** 2 / (n - 1)
    denom     = min(r_corr - 1, c_corr - 1)
    return np.sqrt(phi2_corr / denom) if denom > 0 else 0.0
 
 
def _magnitude_v(v: float) -> str:
    if v >= .4:  return 'Forte'
    if v >= .2:  return 'Moderado'
    if v >= .1:  return 'Pequeno'
    return 'Negligenciável'
 
 
def _magnitude_r(r: float) -> str:
    r = abs(r)
    if r >= .5:  return 'Grande'
    if r >= .3:  return 'Médio'
    if r >= .1:  return 'Pequeno'
    return 'Negligenciável'
 
 
def _magnitude_rho(rho: float) -> str:
    rho = abs(rho)
    if rho >= .5:  return 'Forte'
    if rho >= .3:  return 'Moderada'
    if rho >= .1:  return 'Fraca'
    return 'Negligenciável'
 
 
def wilson_ic(n_sucessos: int, n_total: int, alpha: float = 0.05) -> tuple[float, float]:
    """Intervalo de confiança de Wilson para proporção."""
    if n_total == 0:
        return (np.nan, np.nan)
    z   = norm.ppf(1 - alpha / 2)
    p   = n_sucessos / n_total
    den = 1 + z**2 / n_total
    ctr = (p + z**2 / (2 * n_total)) / den
    mar = z * np.sqrt(p * (1 - p) / n_total + z**2 / (4 * n_total**2)) / den
    return (max(0, ctr - mar), min(1, ctr + mar))
 
 
def fishers_z(r1: float, n1: int, r2: float, n2: int) -> dict:
    """Teste de diferença entre duas correlações de Spearman via transformação z de Fisher."""
    z1  = np.arctanh(r1)
    z2  = np.arctanh(r2)
    se  = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    z   = (z1 - z2) / se
    p   = 2 * (1 - norm.cdf(abs(z)))
    return {'z_Fisher': round(z, 4), 'p_Fisher': round(p, 6)}

df['genero_bin'] = (df['genero'] == 'Feminino').astype(int)
 
df['faixa_salarial_num'] = (
    df['faixa_salarial']
    .map({v: i for i, v in enumerate(FAIXA_SALARIAL_ORDEM)})
    .astype(float)
)
 
df['nivel_num'] = (
    df['nivel']
    .map({v: i for i, v in enumerate(NIVEL_SENIORIDADE_ORDEM)})
    .astype(float)
)
 
df['ano_num']    = df['ano_pesquisa'].astype(int)
df['ano_centro'] = df['ano_num'] - 2021  # centrado para facilitar interpretação
 
variaveis_chi2 = {
    'Nível de Senioridade': 'nivel',
    'Faixa Salarial':       'faixa_salarial',
    'Nível de Ensino':      'nivel_ensino',
    'Cargo':                'cargo',
    'Modalidade':           'modalidade',
    'Situação':             'situacao',
    'Área de Formação':     'area_formacao',
    'Região':               'regiao',
}
 
print(f'N total:     {len(df):,}')
print(f'N Feminino:  {(df["genero"]=="Feminino").sum():,} ({(df["genero"]=="Feminino").mean()*100:.1f}%)')
print(f'N Masculino: {(df["genero"]=="Masculino").sum():,} ({(df["genero"]=="Masculino").mean()*100:.1f}%)')

### 3.1 — H1 / H2 — QUI-QUADRADO + V DE CRAMÉR

In [ ]:
def teste_qui_quadrado(df: pd.DataFrame, col: str, label: str) -> dict:
    sub                    = df.dropna(subset=[col, 'genero'])
    ct                     = pd.crosstab(sub[col], sub['genero'])
    chi2, p, dof, expected = chi2_contingency(ct)
    v                      = cramers_v(ct)
    pct_ok                 = (expected >= 5).mean() * 100
    return {
        'Variável':          label,
        'N':                 len(sub),
        'χ²':                round(chi2, 3),
        'gl':                dof,
        'p-valor':           round(p, 6),
        'Significativo':     'Sim' if p < ALPHA else 'Não',
        'V de Cramér':       round(v, 4),
        'Magnitude':         _magnitude_v(v),
        '% células (exp≥5)': f'{pct_ok:.0f}%',
        'Pressuposto OK':    'Sim' if pct_ok >= 80 else 'Atenção',
    }
 
 
tab_chi2 = pd.DataFrame(
    [teste_qui_quadrado(df, col, label) for label, col in variaveis_chi2.items()]
).set_index('Variável')
 
print('\n=== H1/H2 — Qui-quadrado + V de Cramér ===')
print(tab_chi2.to_string())

### 3.2 — H3 — MANN-WHITNEY U — DIFERENÇA SALARIAL ENTRE GÊNEROS


In [ ]:
def rank_biserial(U: float, n1: int, n2: int) -> float:
    return (2 * U) / (n1 * n2) - 1
 
 
def mann_whitney_salarial(df_sub: pd.DataFrame, label: str = 'Geral') -> dict | None:
    sub  = df_sub.dropna(subset=['faixa_salarial_num', 'genero'])
    masc = sub[sub['genero'] == 'Masculino']['faixa_salarial_num']
    fem  = sub[sub['genero'] == 'Feminino' ]['faixa_salarial_num']
    if len(masc) < 5 or len(fem) < 5:
        return None
 
    U, p = mannwhitneyu(masc, fem, alternative='two-sided')
    r    = rank_biserial(U, len(masc), len(fem))
 
    # IC de Wilson para proporção acima da mediana geral (indicador complementar)
    mediana_geral = sub['faixa_salarial_num'].median()
    n_masc_acima  = (masc > mediana_geral).sum()
    n_fem_acima   = (fem  > mediana_geral).sum()
    ic_masc       = wilson_ic(int(n_masc_acima), len(masc))
    ic_fem        = wilson_ic(int(n_fem_acima),  len(fem))
 
    return {
        'Grupo':               label,
        'N Masc':              len(masc),
        'N Fem':               len(fem),
        'Mediana Masc':        FAIXA_SALARIAL_ORDEM[int(masc.median())],
        'Mediana Fem':         FAIXA_SALARIAL_ORDEM[int(fem.median())],
        'U':                   round(U),
        'p-valor':             round(p, 6),
        'Significativo':       'Sim' if p < ALPHA else 'Não',
        'r (rank-biserial)':   round(r, 4),
        'Direção':             ('Masc > Fem' if r > 0 else 'Fem > Masc' if r < 0 else 'Neutro'),
        'Magnitude':           _magnitude_r(r),
        '% Masc > Mediana':    f'{n_masc_acima/len(masc)*100:.1f}%',
        'IC 95% Wilson Masc':  f'[{ic_masc[0]:.3f}, {ic_masc[1]:.3f}]',
        '% Fem > Mediana':     f'{n_fem_acima/len(fem)*100:.1f}%',
        'IC 95% Wilson Fem':   f'[{ic_fem[0]:.3f}, {ic_fem[1]:.3f}]',
    }
 
 
res_mw = [mann_whitney_salarial(df, 'Brasil — Geral')]
for ano in ANOS:
    res_mw.append(mann_whitney_salarial(df[df['ano_pesquisa'] == ano], str(ano)))
for reg in sorted(df['regiao'].dropna().unique()):
    res_mw.append(mann_whitney_salarial(df[df['regiao'] == reg], f'Região: {reg}'))
 
tab_mw = pd.DataFrame([r for r in res_mw if r]).set_index('Grupo')
print('\n=== H3 — Mann-Whitney U: Diferença Salarial por Gênero ===')
print(tab_mw.to_string())

### 3.3 — H4 — CORRELAÇÃO DE SPEARMAN + FISHER'S Z

In [ ]:
def spearman_por_genero(df: pd.DataFrame, x: str, y: str,
                        label_x: str, label_y: str) -> pd.DataFrame:
    rows = []
    for genero in ['Masculino', 'Feminino', 'Geral']:
        sub = df if genero == 'Geral' else df[df['genero'] == genero]
        sub = sub.dropna(subset=[x, y])
        if len(sub) < 10:
            continue
        rho, p = spearmanr(sub[x], sub[y])
        rows.append({
            'Grupo':                          genero,
            'N':                              len(sub),
            f'ρ ({label_x} × {label_y})':    round(rho, 4),
            'p-valor':                        round(p, 6),
            'Significativo':                  'Sim' if p < ALPHA else 'Não',
            'Magnitude':                      _magnitude_rho(rho),
        })
    df_out = pd.DataFrame(rows).set_index('Grupo')
 
    # Fisher's z: testa se ρ_masculino ≠ ρ_feminino
    if 'Masculino' in df_out.index and 'Feminino' in df_out.index:
        rho_col = f'ρ ({label_x} × {label_y})'
        r_m = df_out.loc['Masculino', rho_col]
        r_f = df_out.loc['Feminino',  rho_col]
        n_m = int(df_out.loc['Masculino', 'N'])
        n_f = int(df_out.loc['Feminino',  'N'])
        fz  = fishers_z(r_m, n_m, r_f, n_f)
        print(f'  Fisher z ({label_x}×{label_y}): '
              f'z={fz["z_Fisher"]}, p={fz["p_Fisher"]} '
              f'→ diferença entre gêneros {"significativa" if fz["p_Fisher"] < ALPHA else "não significativa"}')
    return df_out
 
 
# H4a: nível de senioridade × faixa salarial
tab_sp_nivel_sal = spearman_por_genero(
    df, 'nivel_num', 'faixa_salarial_num', 'Senioridade', 'Salário'
)
 
tab_sp_exp_sal = pd.DataFrame()
if 'tempo_area_dados' in df.columns:
    cats_exp = [
        'Menos de 1 ano', 'de 1 a 2 anos', 'de 2 a 3 anos', 'de 3 a 4 anos',
        'de 4 a 5 anos',  'de 5 a 6 anos', 'de 6 a 7 anos', 'de 7 a 10 anos',
        'Mais de 10 anos',
    ]
    codes = pd.Categorical(df['tempo_area_dados'], categories=cats_exp, ordered=True).codes.astype(float)
    codes[codes == -1] = np.nan
    df['experiencia_num'] = codes
    tab_sp_exp_sal = spearman_por_genero(
        df, 'experiencia_num', 'faixa_salarial_num', 'Experiência', 'Salário'
    )
 
print('\n=== H4 — Spearman: Senioridade × Salário ===')
print(tab_sp_nivel_sal.to_string())
if not tab_sp_exp_sal.empty:
    print('\n=== H4 — Spearman: Experiência × Salário ===')
    print(tab_sp_exp_sal.to_string())
 

### 3.4 — H5 — TENDÊNCIA TEMPORAL (REGRESSÃO LOGÍSTICA + COCHRAN-ARMITAGE)

In [ ]:
def cochran_armitage(df: pd.DataFrame) -> dict:
    """
    Teste de tendência de Cochran-Armitage para proporção feminina por ano.
    Scores ordinais: ano_centro (0, 1, 2, 3 para 2021–2024).
    Referência: Agresti (2002), cap. 7.
    """
    sub   = df.dropna(subset=['genero', 'ano_centro'])
    anos  = sorted(sub['ano_centro'].unique())
    n_k   = np.array([len(sub[sub['ano_centro'] == a])            for a in anos], dtype=float)
    m_k   = np.array([(sub[sub['ano_centro'] == a]['genero_bin']).sum() for a in anos], dtype=float)
    p_hat = m_k.sum() / n_k.sum()
    scores = np.array(anos, dtype=float)
 
    T = np.sum(scores * (m_k - n_k * p_hat))
    V = p_hat * (1 - p_hat) * (
        np.sum(n_k * scores**2) - np.sum(n_k * scores)**2 / n_k.sum()
    )
    z = T / np.sqrt(V)
    p = 2 * (1 - norm.cdf(abs(z)))
    return {
        'T (Cochran-Armitage)': round(T, 4),
        'z':                    round(z, 4),
        'p-valor':              round(p, 6),
        'Tendência':            ('Crescente'  if z > 0 and p < ALPHA
                                 else 'Decrescente' if z < 0 and p < ALPHA
                                 else 'Estável'),
    }
 
 
def regressao_temporal(df_sub: pd.DataFrame, label: str = 'Brasil') -> dict | None:
    sub = df_sub.dropna(subset=['genero_bin', 'ano_centro'])
    if len(sub) < 50:
        return None
    X   = sm.add_constant(sub['ano_centro'])
    mod = sm.Logit(sub['genero_bin'], X).fit(disp=False)
 
    coef  = mod.params['ano_centro']
    p     = mod.pvalues['ano_centro']
    OR    = np.exp(coef)
    ci    = np.exp(mod.conf_int().loc['ano_centro'])
    mcf   = 1 - mod.llf / mod.llnull
 
    return {
        'Grupo':            label,
        'N':                len(sub),
        'β (log-odds/ano)': round(coef, 4),
        'OR':               round(OR, 4),
        'IC 95% OR':        f'[{ci.iloc[0]:.3f}, {ci.iloc[1]:.3f}]',
        'p-valor':          round(p, 6),
        'Significativo':    'Sim' if p < ALPHA else 'Não',
        'McFadden R²':      round(mcf, 4),
        'Tendência':        ('Crescente'  if coef > 0 and p < ALPHA
                             else 'Decrescente' if coef < 0 and p < ALPHA
                             else 'Estável'),
    }
 
 
res_temp = [regressao_temporal(df, 'Brasil — Geral')]
for reg in sorted(df['regiao'].dropna().unique()):
    res_temp.append(regressao_temporal(df[df['regiao'] == reg], f'Região: {reg}'))
 
tab_temp = pd.DataFrame([r for r in res_temp if r]).set_index('Grupo')
print('\n=== H5 — Tendência Temporal da Representatividade Feminina ===')
print(tab_temp.to_string())
 
# Cochran-Armitage global
ca = cochran_armitage(df)
print('\n--- Cochran-Armitage (tendência monotônica 2021–2024) ---')
for k, v in ca.items():
    print(f'  {k}: {v}')

### 3.5 — H6 — ADOÇÃO DE IA GENERATIVA POR GÊNERO (2023–2024)

In [ ]:
df_ia = df[df['ano_pesquisa'].isin(['2023', '2024'])].copy()

def chi2_cramer_ia(df_sub, feat, ano_label):
    sub = df_sub.dropna(subset=[feat, 'genero'])
    if sub['genero'].nunique() < 2:
        return None
    ct = pd.crosstab(sub['genero'], sub[feat])
    if ct.shape[1] < 2:
        return None

    chi2, p, _, _ = chi2_contingency(ct, correction=False)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

    def _pct_ic(index_label):
        if index_label not in ct.index:
            return np.nan, (np.nan, np.nan)
        row   = ct.loc[index_label]
        n_usa = int(row.get(1, 0))
        n_tot = int(row.sum())
        return (n_usa / n_tot * 100 if n_tot > 0 else np.nan,
                wilson_ic(n_usa, n_tot))

    pct_m, ic_m = _pct_ic('Masculino')
    pct_f, ic_f = _pct_ic('Feminino')

    return {
        'Ano':             ano_label,
        'Feature':         IA_FEATURE_LABELS.get(feat, feat),
        '% Masculino':     round(pct_m, 1),
        'IC 95% Masc':     f'[{ic_m[0]:.3f}, {ic_m[1]:.3f}]',
        '% Feminino':      round(pct_f, 1),
        'IC 95% Fem':      f'[{ic_f[0]:.3f}, {ic_f[1]:.3f}]',
        'Gap (M−F) pp':    round(pct_m - pct_f, 1),
        'χ²':              round(chi2, 3),
        'p-valor':         round(p, 6),
        'V de Cramér':     round(v, 4),
        'Sig.':            ('***' if p < 0.001 else '**' if p < 0.01
                            else '*' if p < 0.05 else 'n.s.'),
    }


rows_h6 = []
for ano in ['2023', '2024']:
    sub_ano = df_ia[df_ia['ano_pesquisa'] == ano]
    for feat in IA_FEATURES_MODELO_B:
        r = chi2_cramer_ia(sub_ano, feat, ano)
        if r:
            rows_h6.append(r)

for feat in IA_FEATURES_MODELO_B:
    r = chi2_cramer_ia(df_ia, feat, '2023–2024')
    if r:
        rows_h6.append(r)

df_h6 = pd.DataFrame(rows_h6)

for ano in ['2023', '2024', '2023–2024']:
    print(f'\n── H6 — {ano} ──')
    print(df_h6[df_h6['Ano'] == ano].to_string(index=False))

# Gráfico H6
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, ano in zip(axes, ['2023', '2024']):
    sub = df_h6[df_h6['Ano'] == ano].set_index('Feature')
    x   = np.arange(len(sub))
    w   = 0.35
    ax.bar(x - w/2, sub['% Masculino'], w, color='#517493', alpha=0.87, label='Masculino')
    ax.bar(x + w/2, sub['% Feminino'],  w, color='#64313E', alpha=0.87, label='Feminino')
    for xi, (vm, vf) in enumerate(zip(sub['% Masculino'], sub['% Feminino'])):
        gap = vm - vf
        cor = '#C0392B' if gap > 0 else '#2980B9'
        ax.text(xi, max(vm, vf) + 1.5, f'{gap:+.1f}pp',
                ha='center', fontsize=8, color=cor, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index, rotation=28, ha='right', fontsize=8.5)
    ax.set_ylabel('% que adota / usa' if ano == '2023' else '')
    ax.set_ylim(0, 100)
    ax.set_title(f'Adoção de IA por Gênero — {ano}', fontweight='bold')
    ax.legend(frameon=False, fontsize=9)

plt.suptitle('H6 — Associação entre Gênero e Adoção de IA Generativa',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/3_6_h6_ia_genero.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
resultados = {}
resultados['H1_H2_Chi2']         = tab_chi2
resultados['H3_MannWhitney']     = tab_mw
resultados['H4a_Spearman_Nivel'] = tab_sp_nivel_sal
resultados['H4b_Spearman_Exp']   = tab_sp_exp_sal
resultados['H5_Temporal']        = tab_temp
resultados['H6_IA_Genero']       = df_h6
 
with pd.ExcelWriter('resultados_estatisticos.xlsx') as writer:
    tab_chi2.to_excel(writer,          sheet_name='H1-H2 Chi2 Cramer')
    tab_mw.to_excel(writer,            sheet_name='H3 Mann-Whitney')
    tab_sp_nivel_sal.to_excel(writer,  sheet_name='H4a Spearman Nivel')
    if not tab_sp_exp_sal.empty:
        tab_sp_exp_sal.to_excel(writer, sheet_name='H4b Spearman Exp')
    tab_temp.to_excel(writer,          sheet_name='H5 Tendencia Temporal')
    df_h6.to_excel(writer,             sheet_name='H6 IA Genero', index=False)
 
print('resultados_estatisticos.xlsx salvo')

---
## Seção 4 — Modelagem Preditiva (Machine Learning)



## 4.0 Configuração dos experimentos

In [ ]:
BASE_FEATURES = [
    'nivel_ensino',
    'area_formacao',
    'cargo',
    'tempo_area_dados',
    'tempo_area_ti',
    'modalidade',
    'setor',
    'regiao',
    'situacao',
]

IA_FEATURES = [
    'ia_nao_usa',
    'ia_usa_gratuita',
    'ia_usa_paga_prop',
    'ia_usa_paga_emp',
    'ia_usa_copilot',
    'ia_empresa_prioridade_alta',
    'ia_org_uso_difuso',
]

MODELOS_FEATURES = {
    'A': BASE_FEATURES,
    'B': BASE_FEATURES + IA_FEATURES,
}

In [ ]:
TARGETS = {
    'faixa_salarial':     'multiclass',
    'faixa_salarial_3':   'multiclass',
    'faixa_salarial_bin': 'binario',
    'nivel':              'multiclass',
    'gestor':             'binario',
}

TARGET_LABELS = {
    'faixa_salarial':     'Faixa Salarial (12 classes)',
    'faixa_salarial_3':   'Faixa Salarial (3 classes)',
    'faixa_salarial_bin': 'Faixa Salarial (binário)',
    'nivel':              'Senioridade',
    'gestor':             'Gestor/a',
}

ORDEM_MAP = {
    'faixa_salarial':   FAIXA_SALARIAL_ORDEM,
    'faixa_salarial_3': ['Baixa', 'Média', 'Alta'],
    'nivel':            NIVEL_SENIORIDADE_ORDEM
}

TARGETS_CENTRAIS = ['gestor', 'nivel', 'faixa_salarial_bin']

ESTIMADOR_CENTRAL = 'XGBoost'

## 4.1 Funções de preparação e pipeline

In [ ]:
def prepare_xy(df: pd.DataFrame, target: str, features: list):
    """
    Seleciona features e target, codifica ordinais, descarta NaN no target.
    'nivel' e 'cargo' são excluídos quando target='gestor' (leakage).
    Retorna: X, y (int), meta (genero + ano_pesquisa)
    """
    excluir = ['nivel', 'cargo'] if target == 'gestor' else []
    feats   = [f for f in features if f not in excluir and f in df.columns]

    sub = df[feats + [target, 'genero', 'ano_pesquisa']].copy()

    # Nível de ensino → ordinal numérico
    codes = pd.Categorical(
        sub['nivel_ensino'], categories=NIVEL_ENSINO_ORDEM, ordered=True
    ).codes.astype(float)
    codes[codes == -1] = np.nan
    sub['nivel_ensino'] = codes

    # Target
    if target in ORDEM_MAP:
        codes = pd.Categorical(
            sub[target], categories=ORDEM_MAP[target], ordered=True
        ).codes.astype(float)
        codes[codes == -1] = np.nan
        sub['y'] = codes
    else:
        sub['y'] = pd.to_numeric(sub[target], errors='coerce')

    sub = sub.dropna(subset=['y'])
    return sub[feats].copy(), sub['y'].astype(int), sub[['genero', 'ano_pesquisa']].copy()

In [ ]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Imputação + escalonamento (num) e OHE (cat)."""
    num = X.select_dtypes(include='number').columns.tolist()
    cat = X.select_dtypes(exclude='number').columns.tolist()
    steps = []
    if num:
        steps.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
        ]), num))
    if cat:
        steps.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat))
    return ColumnTransformer(steps, remainder='drop')


In [ ]:
def make_xgb(task_type: str, y_train: pd.Series) -> XGBClassifier:
    """XGBoost configurado para binário ou multiclasse, com scale_pos_weight automático."""
    binary = task_type == 'binario'
    pos_weight = float((y_train == 0).sum()) / max((y_train == 1).sum(), 1) if binary else 1.0
    return XGBClassifier(
        n_estimators=200,
        random_state=SEED,
        verbosity=0,
        eval_metric='logloss' if binary else 'mlogloss',
        scale_pos_weight=pos_weight if binary else 1,
    )

In [ ]:
def make_all_models(task_type: str, y_train: pd.Series) -> dict:
    """Todos os estimadores para validação cruzada interna."""
    kw     = dict(random_state=SEED)
    binary = task_type == 'binario'
    pos_w  = float((y_train == 0).sum()) / max((y_train == 1).sum(), 1) if binary else 1.0
    return {
        'Logistic Regression': LogisticRegression(max_iter=500, class_weight='balanced', **kw),
        'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, **kw),
        'XGBoost':             XGBClassifier(n_estimators=200, verbosity=0, eval_metric='logloss' if binary else 'mlogloss',
                                             scale_pos_weight=pos_w if binary else 1, **kw),
    }

In [ ]:
def run_cv(X: pd.DataFrame, y: pd.Series, task_type: str, cv: int = 5) -> pd.DataFrame:
    """
    Validação cruzada interna (3 algoritmos).
    Serve para transparência metodológica — não seleciona o estimador do experimento central.
    """
    skf  = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    rows = []
    for name, model in make_all_models(task_type, y).items():
        scores = cross_val_score(
            Pipeline([('pre', build_preprocessor(X)), ('clf', model)]),
            X, y, cv=skf, scoring='f1_macro', n_jobs=-1,
        )
        rows.append({'Modelo': name, 'F1_mean': scores.mean(), 'F1_std': scores.std()})
        print(f'    {name:22s}: {scores.mean():.3f} ± {scores.std():.3f}')
    return pd.DataFrame(rows)

## 4.2 Métricas de equidade algorítmica

In [ ]:
def equidade_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                     genero: pd.Series, task_type: str) -> pd.DataFrame:
    """
    Calcula métricas de equidade por gênero:
      - F1-macro, Precision-macro, Recall-macro
      - False Negative Rate (FNR) — targets binários
      - Equal Opportunity Difference (TPR gap) — targets binários
      - Demographic Parity Difference — targets binários
      - Calibration gap (se predict_proba disponível)
    """
    rows = {}
    for g in ['Masculino', 'Feminino']:
        mask = (genero == g).values
        if not mask.any():
            continue
        yt, yp = y_true[mask], y_pred[mask]
        rows[g] = {
            'N':             int(mask.sum()),
            'F1-macro':      round(f1_score(yt, yp, average='macro',    zero_division=0), 4),
            'Precision-mac': round(precision_score(yt, yp, average='macro', zero_division=0), 4),
            'Recall-mac':    round(recall_score(yt, yp, average='macro',    zero_division=0), 4),
        }
        if task_type == 'binario':
            tp = int(((yt == 1) & (yp == 1)).sum())
            fn = int(((yt == 1) & (yp == 0)).sum())
            rows[g]['FNR'] = round(fn / (tp + fn) if (tp + fn) > 0 else np.nan, 4)

    df_eq = pd.DataFrame(rows).T.reset_index().rename(columns={'index': 'Gênero'})

    if task_type == 'binario' and len(df_eq) == 2:
        idx = df_eq.set_index('Gênero')

        # Equal Opportunity Difference (TPR gap = 1 - FNR gap)
        tpr_m = 1 - idx.loc['Masculino', 'FNR']
        tpr_f = 1 - idx.loc['Feminino',  'FNR']
        print(f'  Equal Opportunity Difference (TPR_Masc − TPR_Fem): {tpr_m - tpr_f:+.4f}')

        # Demographic Parity Difference
        dpd = demographic_parity_difference(y_true, y_pred, sensitive_features=genero)
        print(f'  Demographic Parity Difference:                      {dpd:+.4f}')

    if len(df_eq) == 2:
        gap = (df_eq.set_index('Gênero').loc['Masculino', 'F1-macro']
             - df_eq.set_index('Gênero').loc['Feminino',  'F1-macro'])
        label = ('→ favorece Masculino' if gap > 0.03
                 else '→ favorece Feminino' if gap < -0.03
                 else '→ equilibrado')
        print(f'  Gap F1 (Masc − Fem): {gap:+.4f}  {label}')

    return df_eq

## 4.3 Função central de treinamento e avaliação

In [ ]:
def treinar_avaliar(target: str, task_type: str,
                    df_train: pd.DataFrame, df_test: pd.DataFrame,
                    modelo_key: str = 'A',
                    run_cv_interno: bool = True) -> dict:
    """
    Treina o estimador central (XGBoost fixo) com as features do Modelo A ou B.
    CV interno é opcional (para transparência, não para seleção).

    Retorna dict com: target, label, pipe, X_tr, X_te, y_te, y_pred, meta, df_equidade
    """
    features = MODELOS_FEATURES[modelo_key]
    tag      = f'{TARGET_LABELS[target]} [Modelo {modelo_key}]'
    print(f'\n{"="*60}\n  {tag}\n{"="*60}')

    X_tr, y_tr, _    = prepare_xy(df_train, target, features)
    X_te, y_te, meta = prepare_xy(df_test,  target, features)
    print(f'  Treino: {len(X_tr):,} | Teste: {len(X_te):,} | Classes: {sorted(y_tr.unique())}')

    # Validação cruzada interna 
    if run_cv_interno:
        print(f'\n  ── CV interno (validação de estabilidade) ──')
        run_cv(X_tr, y_tr, task_type)

    # Treino com estimador fixo
    print(f'\n  ── Estimador central: {ESTIMADOR_CENTRAL} ──')
    pipe = Pipeline([
        ('pre', build_preprocessor(X_tr)),
        ('clf', make_xgb(task_type, y_tr)),
    ])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)

    print(classification_report(y_te, y_pred, zero_division=0))

    # Equidade
    print('\n  ── Métricas de Equidade ──')
    df_eq = equidade_metrics(y_te.values, y_pred, meta['genero'], task_type)
    print(df_eq.to_string(index=False))

    return dict(
        target=target, label=tag, modelo_key=modelo_key,
        pipe=pipe, estimador=ESTIMADOR_CENTRAL,
        X_tr=X_tr, X_te=X_te,
        y_te=y_te, y_pred=y_pred,
        meta=meta, df_equidade=df_eq,
    )

## 4.4 Experimento central: Modelo A vs Modelo B

In [ ]:
df_train = pd.read_parquet('data/df_train.parquet')
df_test  = pd.read_parquet('data/df_test.parquet')

# Recriar targets salariais agrupados (caso os parquets não os contenham)
df_train = criar_targets_salariais(df_train)
df_test  = criar_targets_salariais(df_test)

print(f'Treino: {len(df_train):,} | Teste: {len(df_test):,}')

# Targets centrais: gestor, senioridade, salário binário
results_A, results_B = {}, {}

for target in TARGETS_CENTRAIS:
    results_A[target] = treinar_avaliar(target, TARGETS[target], df_train, df_test, modelo_key='A')
    results_B[target] = treinar_avaliar(target, TARGETS[target], df_train, df_test, modelo_key='B')


## 4.5 Tabela de ablação: ΔF1 = B − A

In [ ]:
def tabela_ablacao(results_A: dict, results_B: dict,
                   targets: list = TARGETS_CENTRAIS) -> pd.DataFrame:
    rows = []
    for target in targets:
        ra, rb   = results_A[target], results_B[target]
        f1_a     = f1_score(ra['y_te'], ra['y_pred'], average='macro', zero_division=0)
        f1_b     = f1_score(rb['y_te'], rb['y_pred'], average='macro', zero_division=0)
        delta    = f1_b - f1_a
        rows.append({
            'Desfecho':   TARGET_LABELS[target],
            'Estimador':  ESTIMADOR_CENTRAL,
            'F1-macro A': round(f1_a, 4),
            'F1-macro B': round(f1_b, 4),
            'ΔF1 (B−A)':  round(delta, 4),
            'Δ%':         round(delta / f1_a * 100 if f1_a > 0 else np.nan, 1),
            'Direção':    ('↑ IA ajuda' if delta > 0.01
                           else '↓ IA atrapalha' if delta < -0.01
                           else '→ Neutro'),
        })
    return pd.DataFrame(rows).set_index('Desfecho')


tab_ablacao = tabela_ablacao(results_A, results_B)
print('\n=== ABLAÇÃO: Ganho Incremental das Variáveis de IA ===')
print(tab_ablacao.to_string())


def plot_ablacao(tab: pd.DataFrame, fname: str = 'ablacao_ia.png'):
    labels = tab.index.tolist()
    x, w   = np.arange(len(labels)), 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w/2, tab['F1-macro A'], w, label='Modelo A (sem IA)',  color='#4C8FBF', alpha=0.85)
    ax.bar(x + w/2, tab['F1-macro B'], w, label='Modelo B (com IA)', color='#F4A43B', alpha=0.85)

    for xi, (fa, fb) in enumerate(zip(tab['F1-macro A'], tab['F1-macro B'])):
        delta = fb - fa
        cor   = '#2ECC71' if delta > 0.005 else ('#E74C3C' if delta < -0.005 else '#7F8C8D')
        ax.annotate(f'Δ={delta:+.3f}', xy=(xi + w/2, fb),
                    xytext=(0, 6), textcoords='offset points',
                    ha='center', fontsize=9, color=cor, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel('F1-macro (conjunto de teste 2024)')
    ax.set_ylim(0, min(1.0, tab[['F1-macro A', 'F1-macro B']].max().max() + 0.15))
    ax.set_title(f'Ablação — {ESTIMADOR_CENTRAL}: Contribuição Incremental das Variáveis de IA',
                 fontweight='bold', fontsize=12)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'figures/{fname}')


plot_ablacao(tab_ablacao)

## 4.6 Equidade consolidada (A e B, todos os targets centrais)

In [ ]:
frames = []
for target in TARGETS_CENTRAIS:
    for key, results in [('A', results_A), ('B', results_B)]:
        df_eq = results[target]['df_equidade'].copy()
        df_eq.insert(0, 'Modelo',    key)
        df_eq.insert(0, 'Desfecho',  TARGET_LABELS[target])
        frames.append(df_eq)

df_eq_all = pd.concat(frames, ignore_index=True)
print('\n=== AVALIAÇÃO DE EQUIDADE CONSOLIDADA ===')
print(df_eq_all.to_string(index=False))
df_eq_all.to_excel('equidade_consolidada.xlsx', index=False)

## 4.7 Experimento de sensibilidade: granularidade do target salarial


In [ ]:
print('\n=== Sensibilidade: Granularidade do Target Salarial ===')
targets_sal = ['faixa_salarial_bin', 'faixa_salarial_3', 'faixa_salarial']
rows_sal    = []

for t in targets_sal:
    r  = treinar_avaliar(t, TARGETS[t], df_train, df_test, modelo_key='B', run_cv_interno=False)
    f1 = f1_score(r['y_te'], r['y_pred'], average='macro', zero_division=0)
    rows_sal.append({
        'Versão':      TARGET_LABELS[t],
        'Nº classes':  len(set(r['y_te'])),
        'F1-macro':    round(f1, 4),
    })

print(pd.DataFrame(rows_sal).to_string(index=False))

---
## Seção 5 — Explicabilidade com SHAP

## 5.0 Utilitários

In [ ]:
def clean_feat_name(name: str, max_len: int = 40) -> str:
    name = name.replace('cat__', '').replace('num__', '')
    return name[:max_len] + '...' if len(name) > max_len else name

In [ ]:
def compute_shap(pipe, X: pd.DataFrame, max_samples: int = 200):
    """
    Calcula valores SHAP para uma amostra de X.
    Retorna (mean_abs_norm, feat_names).
    """
    pre = pipe.named_steps['pre']
    clf = pipe.named_steps['clf']

    idx = np.random.RandomState(SEED).choice(len(X), min(max_samples, len(X)), replace=False)
    X_t = pre.transform(X.iloc[idx])

    try:
        feat_names = np.array(pre.get_feature_names_out())
    except Exception:
        feat_names = np.array([f'f{i}' for i in range(X_t.shape[1])])

    sv = shap.TreeExplainer(
        clf, feature_perturbation='tree_path_dependent'
    ).shap_values(X_t, check_additivity=False)

    if isinstance(sv, list):
        mean_abs = np.mean([np.abs(s) for s in sv], axis=0).mean(axis=0)
    else:
        sv = np.array(sv)
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        mean_abs = np.abs(sv).mean(axis=0)

    total = mean_abs.sum()
    if total > 0:
        mean_abs = mean_abs / total

    return mean_abs, feat_names


In [ ]:
def plot_shap_bar(mean_shap, feat_names, top_n: int = 15,
                  title: str = 'SHAP', fname: str = 'shap.png',
                  highlight_ia: bool = False):
    top_idx = np.argsort(mean_shap)[::-1][:top_n]
    names   = [clean_feat_name(feat_names[i]) for i in top_idx]
    vals    = mean_shap[top_idx]
    colors  = (['#F4A43B' if 'ia_' in feat_names[i] else '#4C8FBF' for i in top_idx]
               if highlight_ia else ['#4C8FBF'] * top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(names[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel('Mean |SHAP value| (normalizado)')
    ax.set_title(title, fontweight='bold', loc='left')
    fig.subplots_adjust(left=0.35)
    if highlight_ia:
        ax.legend(handles=[
            mpatches.Patch(color='#F4A43B', label='Feature de IA generativa'),
            mpatches.Patch(color='#4C8FBF', label='Feature estrutural'),
        ], loc='lower right', frameon=False)
    plt.savefig(f'figures/{fname}', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  figures/{fname}')

## 5.1 SHAP global: Modelo A vs Modelo B

In [ ]:
shap_cache = {}

for target in TARGETS_CENTRAIS:
    safe = target.replace(' ', '_')
    for key, results in [('A', results_A), ('B', results_B)]:
        try:
            r       = results[target]
            ms, fn  = compute_shap(r['pipe'], r['X_te'])
            shap_cache[f'{key}_{target}'] = (ms, fn)
            plot_shap_bar(ms, fn,
                          title=f'SHAP — {TARGET_LABELS[target]} · Modelo {key}',
                          fname=f'shap_{key}_{safe}.png',
                          highlight_ia=(key == 'B'))
        except Exception as e:
            print(f'  SHAP Modelo {key}/{target}: {e}')

    # Comparativo A vs B lado a lado
    key_a, key_b = f'A_{target}', f'B_{target}'
    if key_a not in shap_cache or key_b not in shap_cache:
        continue

    ms_a, fn_a = shap_cache[key_a]
    ms_b, fn_b = shap_cache[key_b]

    def top_n(ms, fn, n=12):
        idx = np.argsort(ms)[::-1][:n]
        return [clean_feat_name(fn[i]) for i in idx], ms[idx], idx

    names_a, vals_a, idx_a = top_n(ms_a, fn_a)
    names_b, vals_b, idx_b = top_n(ms_b, fn_b)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, names, vals, raw_idx, fn_raw, subtitle, hl in [
        (axes[0], names_a, vals_a, idx_a, fn_a, 'Modelo A (sem IA)', False),
        (axes[1], names_b, vals_b, idx_b, fn_b, 'Modelo B (com IA)', True),
    ]:
        colors = (['#F4A43B' if 'ia_' in fn_raw[i] else '#4C8FBF' for i in raw_idx]
                  if hl else ['#4C8FBF'] * len(names))
        ax.barh(names[::-1], vals[::-1], color=colors[::-1])
        ax.set_xlabel('Mean |SHAP value| (normalizado)')
        ax.set_title(subtitle, fontweight='bold')

    fig.suptitle(f'SHAP Comparativo — {TARGET_LABELS[target]}', fontsize=13, fontweight='bold')
    fig.subplots_adjust(left=0.25, wspace=0.6)
    plt.savefig(f'figures/shap_comp_{safe}.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'  figures/shap_comp_{safe}.png')

## 5.2 SHAP por gênero + correlação de Spearman entre rankings

In [ ]:
def compute_shap_genero(pipe, X_te: pd.DataFrame, meta: pd.DataFrame,
                        max_samples: int = 200) -> dict:
    """SHAP separado por gênero. Retorna {genero: (mean_abs, feat_names)}."""
    resultado = {}
    for genero in ['Masculino', 'Feminino']:
        mask  = (meta['genero'] == genero).values
        X_sub = X_te[mask]
        if len(X_sub) < 20:
            print(f'  Aviso: n={len(X_sub)} para {genero} — SHAP ignorado')
            continue
        ms, fn = compute_shap(pipe, X_sub, max_samples=max_samples)
        resultado[genero] = (ms, fn)
    return resultado

In [ ]:
def spearman_rankings(ms_m, fn_m, ms_f, fn_f, top_n: int = 15) -> pd.DataFrame | None:
    """Correlação de Spearman entre rankings de SHAP por gênero."""
    def top_feats(ms, fn, n):
        idx = np.argsort(ms)[::-1][:n]
        return [clean_feat_name(fn[i]) for i in idx], np.sort(ms)[::-1][:n]

    feats_m, vals_m = top_feats(ms_m, fn_m, top_n)
    feats_f, vals_f = top_feats(ms_f, fn_f, top_n)
    comum           = [f for f in feats_m if f in feats_f]

    if len(comum) < 5:
        print(f'  Menos de 5 features comuns ({len(comum)}) — Spearman não calculado')
        return None

    rho, p = spearmanr(
        [feats_m.index(f) + 1 for f in comum],
        [feats_f.index(f) + 1 for f in comum],
    )
    nivel = ('Alta concordância' if rho > 0.7
             else 'Concordância moderada' if rho > 0.4
             else 'Baixa concordância')
    print(f'  ρ Spearman (n={len(comum)} features comuns): {rho:.3f}  p={p:.4f}  → {nivel}')

    df_m = pd.DataFrame({'Rank': range(1, top_n+1), 'Feature_Masc': feats_m, 'SHAP_Masc': vals_m.round(4)})
    df_f = pd.DataFrame({'Rank': range(1, top_n+1), 'Feature_Fem':  feats_f, 'SHAP_Fem':  vals_f.round(4)})
    return df_m.merge(df_f, on='Rank')

In [ ]:
def plot_shap_genero(shap_g: dict, target: str, top_n: int = 12):
    if 'Masculino' not in shap_g or 'Feminino' not in shap_g:
        print(f'  Dados insuficientes — {target}')
        return

    ms_m, fn_m = shap_g['Masculino']
    ms_f, fn_f = shap_g['Feminino']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, (ms, fn), titulo in zip(axes, [(ms_m, fn_m), (ms_f, fn_f)], ['Masculino', 'Feminino']):
        idx    = np.argsort(ms)[::-1][:top_n]
        names  = [clean_feat_name(fn[i]) for i in idx]
        vals   = ms[idx]
        cor_base = '#517493' if titulo == 'Masculino' else '#64313E'
        colors = ['#F4A43B' if 'ia_' in fn[i] else cor_base for i in idx]
        ax.barh(names[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
        ax.set_xlabel('Mean |SHAP value| (normalizado)')
        ax.set_title(titulo, fontweight='bold', fontsize=12)

    fig.suptitle(f'SHAP por Gênero — {TARGET_LABELS[target]}', fontsize=13, fontweight='bold')
    fig.legend(handles=[
        mpatches.Patch(color='#F4A43B', label='Feature de IA generativa'),
        mpatches.Patch(color='#517493', label='Feature estrutural (Masculino)'),
        mpatches.Patch(color='#64313E', label='Feature estrutural (Feminino)'),
    ], loc='lower center', ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.04))
    fig.subplots_adjust(left=0.22, wspace=0.55, bottom=0.12)
    fname = f'shap_genero_{target}.png'
    plt.savefig(f'figures/{fname}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  figures/{fname}')



In [ ]:
# Execução
for target in TARGETS_CENTRAIS:
    print(f'\n{"="*55}\n  SHAP por Gênero — {TARGET_LABELS[target]}\n{"="*55}')
    rb     = results_B[target]
    shap_g = compute_shap_genero(rb['pipe'], rb['X_te'], rb['meta'])

    if 'Masculino' in shap_g and 'Feminino' in shap_g:
        ms_m, fn_m = shap_g['Masculino']
        ms_f, fn_f = shap_g['Feminino']
        comp = spearman_rankings(ms_m, fn_m, ms_f, fn_f)
        if comp is not None:
            print(comp.to_string(index=False))
        plot_shap_genero(shap_g, target)


In [ ]:
# Seção 5 — SHAP Beeswarm: Gestor/a Modelo B
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def compute_shap_values_raw(pipe, X: pd.DataFrame, max_samples: int = 500):
    """
    Retorna shap_values brutos (não agregados) e feature names.
    Necessário para beeswarm.
    """
    pre = pipe.named_steps['pre']
    clf = pipe.named_steps['clf']

    idx = np.random.RandomState(SEED).choice(len(X), min(max_samples, len(X)), replace=False)
    X_sample = X.iloc[idx]
    X_t = pre.transform(X_sample)

    try:
        feat_names = np.array(pre.get_feature_names_out())
    except Exception:
        feat_names = np.array([f'f{i}' for i in range(X_t.shape[1])])

    explainer = shap.TreeExplainer(clf, feature_perturbation='tree_path_dependent')
    shap_out  = explainer.shap_values(X_t, check_additivity=False)

    # Binário: shap_values é lista [classe_0, classe_1] — pega classe positiva
    if isinstance(shap_out, list):
        sv = shap_out[1]
    else:
        sv = np.array(shap_out)
        if sv.ndim == 3:
            sv = sv[:, :, 1]

    return sv, X_t, feat_names, X_sample.index


def plot_beeswarm_gestor(pipe, X_te, meta, top_n=15, max_samples=500):
    """
    Beeswarm SHAP para gestor/a — Modelo B.
    Features de IA destacadas em laranja no eixo Y.
    """
    sv, X_t, feat_names, sample_idx = compute_shap_values_raw(pipe, X_te, max_samples)

    # Seleciona top N features por mean |SHAP|
    mean_abs = np.abs(sv).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:top_n]

    sv_top    = sv[:, top_idx]
    X_top     = X_t[:, top_idx]
    names_top = np.array([clean_feat_name(feat_names[i]) for i in top_idx])

    # Cria Explanation object para o beeswarm nativo do SHAP
    exp = shap.Explanation(
        values          = sv_top,
        data            = X_top,
        feature_names   = names_top,
    )

    # --- Figura ---
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.plots.beeswarm(exp, max_display=top_n, show=False, color_bar=True)

    # Destaca features de IA no eixo Y em laranja
    ax = plt.gca()
    for label in ax.get_yticklabels():
        txt = label.get_text()
        # verifica se a feature original (antes do clean) contém 'ia_'
        orig_matches = [feat_names[i] for i in top_idx
                        if clean_feat_name(feat_names[i]) == txt and 'ia_' in feat_names[i]]
        if orig_matches:
            label.set_color('#E07B00')
            label.set_fontweight('bold')

    ax.set_title(
        'SHAP Beeswarm — Gestor/a (Modelo B)\n'
        'Impacto individual de cada feature na predição',
        fontweight='bold', fontsize=12, loc='left'
    )
    ax.set_xlabel('Valor SHAP (impacto na predição de ser gestor/a)')

    # Legenda manual para features de IA
    leg = [
        mpatches.Patch(color='#E07B00', label='Feature de IA generativa'),
        mpatches.Patch(color='#4C8FBF', label='Feature estrutural'),
    ]
    ax.legend(handles=leg, loc='lower right', frameon=False, fontsize=9)

    plt.tight_layout()
    fname = 'figures/shap_beeswarm_gestor_B.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Salvo: {fname}')


# Execução
rb_gestor = results_B['gestor']
plot_beeswarm_gestor(rb_gestor['pipe'], rb_gestor['X_te'], rb_gestor['meta'])

In [ ]:
def plot_beeswarm_gestor_genero(pipe, X_te, meta, top_n=15, max_samples=400):

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))  # remove sharey aqui
    x_min, x_max = 0, 0
    shap_data = {}

    for genero in ['Masculino', 'Feminino']:
        mask  = (meta['genero'] == genero).values
        X_sub = X_te[mask]
        sv, X_t, feat_names, _ = compute_shap_values_raw(pipe, X_sub, max_samples)

        mean_abs = np.abs(sv).mean(axis=0)
        top_idx  = np.argsort(mean_abs)[::-1][:top_n]

        sv_top    = sv[:, top_idx]
        X_top     = X_t[:, top_idx]
        names_top = np.array([clean_feat_name(feat_names[i]) for i in top_idx])

        shap_data[genero] = {
            'sv_top': sv_top, 'X_top': X_top,
            'names_top': names_top,
            'feat_names_orig': feat_names, 'top_idx': top_idx,
        }
        x_min = min(x_min, sv_top.min())
        x_max = max(x_max, sv_top.max())

    cores_titulo = {'Masculino': '#517493', 'Feminino': '#64313E'}
    x_lim = max(abs(x_min), abs(x_max)) * 1.15

    for ax, genero in zip(axes, ['Masculino', 'Feminino']):
        d = shap_data[genero]
        exp = shap.Explanation(
            values        = d['sv_top'],
            data          = d['X_top'],
            feature_names = d['names_top'],
        )

        plt.sca(ax)
        shap.plots.beeswarm(
            exp,
            max_display = top_n,
            show        = False,
            color_bar   = False,   
        )

        # Destaca features de IA
        for label in ax.get_yticklabels():
            txt = label.get_text()
            is_ia = any(
                'ia_' in d['feat_names_orig'][i]
                for i in d['top_idx']
                if clean_feat_name(d['feat_names_orig'][i]) == txt
            )
            if is_ia:
                label.set_color('#E07B00')
                label.set_fontweight('bold')

        ax.set_title(genero, fontweight='bold', fontsize=13,
                     color=cores_titulo[genero], pad=10)
        ax.set_xlabel('Valor SHAP', fontsize=10)
        ax.set_xlim(-x_lim, x_lim)

        # Remove ylabel do subplot direito
        if genero == 'Feminino':
            ax.set_ylabel('')
            ax.tick_params(axis='y', which='both', left=False, labelleft=False)

    # Color bar única à direita
    sm = plt.cm.ScalarMappable(
        cmap=plt.get_cmap('coolwarm'),
        norm=plt.Normalize(vmin=0, vmax=1)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[1], fraction=0.03, pad=0.02)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.set_label('Feature value', fontsize=9)

    fig.suptitle(
        'SHAP Beeswarm por Gênero — Gestor/a (Modelo B)\n'
        'As mesmas features explicam liderança para homens e mulheres?',
        fontweight='bold', fontsize=13, y=1.02
    )

    leg = [
        mpatches.Patch(color='#E07B00', label='Feature de IA generativa'),
        mpatches.Patch(color='#4C8FBF', label='Feature estrutural'),
    ]
    fig.legend(handles=leg, loc='lower center', ncol=2,
               frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.03))

    plt.tight_layout()
    fname = 'figures/shap_beeswarm_gestor_B_genero.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Salvo: {fname}')

rb_gestor = results_B['gestor']
plot_beeswarm_gestor_genero(rb_gestor['pipe'], rb_gestor['X_te'], rb_gestor['meta'])